# 몽글마을 LLM 의미/다양성 추가 평가 노트북

이 노트북은 이미 생성된 `data/evaluation/raw_outputs*.jsonl` 파일만 사용해서 아래 4가지 평가를 추가로 계산합니다.

1. Embedding Cosine Similarity
2. BERTScore-F1
3. Pairwise Cosine Similarity
4. Distinct-1 / Distinct-2

모델을 다시 호출하지 않으며, 실제 사용자 데이터도 생성하지 않습니다. 첫 실행 시에는 임베딩 모델과 BERTScore 모델 다운로드가 필요할 수 있습니다.

## 1. 설치 안내

로컬 환경에 패키지가 없다면 아래 셀의 주석을 해제해서 설치하세요. 첫 실행은 모델 다운로드 때문에 시간이 걸릴 수 있습니다.

In [1]:
# 필요할 때만 주석을 해제해서 실행하세요.
# !pip install -U pandas numpy scikit-learn sentence-transformers bert-score torch transformers tqdm matplotlib seaborn

## 2. Imports

In [2]:
import json
import math
import re
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 3. 경로 / 설정

In [3]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent]
    for path in candidates:
        if (path / "AGENTS.md").exists() and (path / "data").exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
EVAL_DIR = DATA_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RAW_OUTPUT_PATTERN = "raw_outputs*.jsonl"

# 다국어/한국어 문장 임베딩용. 더 큰 모델로 바꾸면 품질은 좋아질 수 있지만 실행이 느려집니다.
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# BERTScore용 다국어 모델. 한국어 비교에 사용할 수 있고 상대적으로 가볍습니다.
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
BERTSCORE_BATCH_SIZE = 16

SEMANTIC_SCORE_PATH = EVAL_DIR / "semantic_similarity_scores.csv"
PAIRWISE_SCORE_PATH = EVAL_DIR / "pairwise_cosine_similarity.csv"
PAIRWISE_SUMMARY_PATH = EVAL_DIR / "pairwise_cosine_summary.csv"
DISTINCT_SCORE_PATH = EVAL_DIR / "distinct_scores.csv"
MODEL_SUMMARY_PATH = EVAL_DIR / "semantic_distinct_model_summary.csv"
TASK_SUMMARY_PATH = EVAL_DIR / "semantic_distinct_task_summary.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_DIR:", EVAL_DIR)

PROJECT_ROOT: /Users/areum/skn/final
EVAL_DIR: /Users/areum/skn/final/data/evaluation


## 4. Raw output 로드

`raw_outputs*.jsonl` 전체를 읽습니다. 기존 결과 파일은 수정하지 않습니다.

In [4]:
raw_files = sorted(EVAL_DIR.glob(RAW_OUTPUT_PATTERN))
if not raw_files:
    raise FileNotFoundError(f"raw output 파일을 찾지 못했습니다: {EVAL_DIR / RAW_OUTPUT_PATTERN}")

records: List[Dict[str, Any]] = []
for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL 파싱 실패: {path}:{line_no}") from exc
            row["source_file"] = path.name
            records.append(row)

raw_df = pd.DataFrame(records)
print(f"raw files: {len(raw_files)}")
print(f"records: {len(raw_df)}")
display(raw_df.groupby(["model_key", "task"]).size().unstack(fill_value=0))

raw files: 6
records: 350


task,character_message_generation,feed_post_generation,planner_chatbot,quest_generation,todo_decomposition
model_key,,,,,
gemma-3-4b-it,10,10,10,10,10
kanana-nano-2.1b-vllm,10,10,10,10,10
llama-3b,10,10,10,10,10
midm-base,10,10,10,10,10
midm-mini,10,10,10,10,10
qwen-3b,10,10,10,10,10
qwen-7b,10,10,10,10,10


## 5. 평가용 텍스트 추출

구조화 JSON 전체를 문자열로 비교하면 중괄호, 필드명, 코드블록 때문에 점수가 왜곡될 수 있습니다. 그래서 사용자에게 실제로 보이는 텍스트 필드를 우선 추출합니다.

- 우선 사용: `parsed_output`
- 파싱 실패/빈 값: `raw_output` 정리본 사용
- 비교 기준: `expected`

TODO/플래너처럼 구조화된 출력은 `title`, `tag`, `date`, `time`, `summary`, `description`, `message` 등을 모아 비교합니다.

In [5]:
TEXT_FIELD_PRIORITY = {
    "quest",
    "post",
    "message",
    "summary",
    "title",
    "description",
    "tag",
    "date",
    "time",
}

SKIP_FIELD_NAMES = {
    "sample_id",
    "character_id",
    "response_type",
    "missing_fields",
    "expected_todo_count",
    "estimated_minutes",
}

CODE_FENCE_RE = re.compile(r"```(?:json)?|```", re.IGNORECASE)
SPACE_RE = re.compile(r"\s+")


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text)
    text = CODE_FENCE_RE.sub(" ", text)
    text = text.replace("\n", " ")
    text = SPACE_RE.sub(" ", text).strip()
    return text


def collect_text_values(value: Any, parent_key: Optional[str] = None) -> List[str]:
    if value is None:
        return []
    if isinstance(value, dict):
        chunks: List[str] = []
        # 사용자 노출 가능성이 높은 필드를 먼저 모읍니다.
        for key in TEXT_FIELD_PRIORITY:
            if key in value:
                chunks.extend(collect_text_values(value[key], key))
        for key, child in value.items():
            if key in TEXT_FIELD_PRIORITY or key in SKIP_FIELD_NAMES:
                continue
            chunks.extend(collect_text_values(child, key))
        return chunks
    if isinstance(value, list):
        chunks = []
        for item in value:
            chunks.extend(collect_text_values(item, parent_key))
        return chunks
    if isinstance(value, (str, int, float)) and not isinstance(value, bool):
        text = normalize_text(value)
        if not text:
            return []
        if parent_key in SKIP_FIELD_NAMES:
            return []
        return [text]
    return []


def extract_structured_text(value: Any) -> str:
    chunks = collect_text_values(value)
    return normalize_text(" ".join(chunks))


def extract_candidate_text(row: pd.Series) -> Tuple[str, str]:
    parsed = row.get("parsed_output")
    parse_error = row.get("parse_error")
    structured = extract_structured_text(parsed)
    if structured and not parse_error:
        return structured, "parsed_output"
    raw = normalize_text(row.get("raw_output", ""))
    return raw, "raw_output"


def extract_reference_text(row: pd.Series) -> str:
    return extract_structured_text(row.get("expected"))

texts = []
for _, row in raw_df.iterrows():
    candidate_text, candidate_source = extract_candidate_text(row)
    reference_text = extract_reference_text(row)
    texts.append({
        "model_key": row.get("model_key"),
        "model_id": row.get("model_id"),
        "task": row.get("task"),
        "sample_id": row.get("sample_id"),
        "source_file": row.get("source_file"),
        "candidate_source": candidate_source,
        "candidate_text": candidate_text,
        "reference_text": reference_text,
        "candidate_len": len(candidate_text),
        "reference_len": len(reference_text),
        "parse_error": row.get("parse_error"),
    })

text_df = pd.DataFrame(texts)
text_df["has_candidate"] = text_df["candidate_text"].str.len() > 0
text_df["has_reference"] = text_df["reference_text"].str.len() > 0

print(text_df[["has_candidate", "has_reference"]].mean())
display(text_df.head(10))

has_candidate    1.0
has_reference    1.0
dtype: float64


,model_key,model_id,task,sample_id,source_file,candidate_source,candidate_text,reference_text,candidate_len,reference_len,parse_error,has_candidate,has_reference
0,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-001,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,운동하기 프로젝트 기획서 쓰기 장보기,오늘 운동 운동하기 오늘 프로젝트 프로젝트 기획서 작성하기 오늘 생활 장보기,20,42,None,True,True
1,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-002,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 09:00 발표 자료 수정,내일 프로젝트 오전 발표 자료 수정하기 내일 회의 오후 5시 팀 회의 참석하기,25,43,None,True,True
2,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-003,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-06-01 학습 2시간 자격증 강의 3개 듣기,이번 주 공부 자격증 강의 3개 듣기 이번 주 공부 기출 1회 풀기,30,37,None,True,True
3,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-004,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 일정 09:00 방 청소,오늘 집안일 빨래 돌리기 오늘 집안일 방 청소하기,24,27,None,True,True
4,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-005,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,금요일 보고서 초안 작성 금요일 참고 자료 정리,금요일까지 업무 보고서 초안 작성하기 금요일까지 업무 참고자료 정리하기,26,39,None,True,True
5,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-006,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 19:00 저녁 7시 산책,운동 저녁 7시 산책하기 요리 샐러드 준비하기,25,25,None,True,True
6,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-007,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 깃 커밋 2024-05-27 버그 목록 정리 2024-05-27...,오늘 개발 깃 커밋하기 오늘 개발 버그 목록 정리하기 오늘 개발 테스트 돌리기,54,43,None,True,True
7,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-008,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 독서 책 30쪽 읽기,내일 독서 책 30쪽 읽기 내일 독서 독서 메모 남기기,22,30,None,True,True
8,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-009,raw_outputs-llama-3b_gemma3-4b.jsonl,parsed_output,2024-05-27 운동 10:00 오전 10시,건강 오전 10시 물 마시기 운동 오전 10시 스트레칭하기,26,32,None,True,True
9,llama-3b,meta-llama/Llama-3.2-3B-Instruct,todo_decomposition,TODO-SAMPLE-010,raw_outputs-llama-3b_gemma3-4b.jsonl,raw_output,"{""todos"": [{""title"": ""책상 정리하기"", ""tag"": ""일정"", ""...",정리 책상 정리하기 업무 메일 확인하기,136,21,"Expecting ',' delimiter: line 1 column 137 (ch...",True,True


## 6. Embedding Cosine Similarity

생성 텍스트와 기준 텍스트를 같은 임베딩 공간에 올린 뒤 cosine similarity를 계산합니다.

In [6]:
from sentence_transformers import SentenceTransformer

valid_df = text_df[text_df["has_candidate"] & text_df["has_reference"]].copy().reset_index(drop=True)
print(f"valid rows for reference-based metrics: {len(valid_df)} / {len(text_df)}")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

candidate_embeddings = embedding_model.encode(
    valid_df["candidate_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
reference_embeddings = embedding_model.encode(
    valid_df["reference_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

valid_df["embedding_cosine_similarity"] = np.sum(candidate_embeddings * reference_embeddings, axis=1)
display(valid_df[["model_key", "task", "sample_id", "embedding_cosine_similarity"]].head())

valid rows for reference-based metrics: 350 / 350


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

,model_key,task,sample_id,embedding_cosine_similarity
0,llama-3b,todo_decomposition,TODO-SAMPLE-001,0.782166
1,llama-3b,todo_decomposition,TODO-SAMPLE-002,0.573381
2,llama-3b,todo_decomposition,TODO-SAMPLE-003,0.801901
3,llama-3b,todo_decomposition,TODO-SAMPLE-004,0.720374
4,llama-3b,todo_decomposition,TODO-SAMPLE-005,0.902780


## 7. BERTScore-F1

BERTScore는 생성 텍스트(candidate)와 기준 텍스트(reference)의 contextual token similarity를 계산합니다.

In [7]:
from bert_score import score as bert_score

P, R, F1 = bert_score(
    cands=valid_df["candidate_text"].tolist(),
    refs=valid_df["reference_text"].tolist(),
    model_type=BERTSCORE_MODEL_TYPE,
    lang="ko",
    batch_size=BERTSCORE_BATCH_SIZE,
    rescale_with_baseline=False,
    verbose=True,
)

valid_df["bertscore_precision"] = P.detach().cpu().numpy()
valid_df["bertscore_recall"] = R.detach().cpu().numpy()
valid_df["bertscore_f1"] = F1.detach().cpu().numpy()

display(valid_df[["model_key", "task", "sample_id", "bertscore_f1"]].head())

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/24 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/22 [00:00<?, ?it/s]

done in 12.31 seconds, 28.44 sentences/sec


,model_key,task,sample_id,bertscore_f1
0,llama-3b,todo_decomposition,TODO-SAMPLE-001,0.837689
1,llama-3b,todo_decomposition,TODO-SAMPLE-002,0.674748
2,llama-3b,todo_decomposition,TODO-SAMPLE-003,0.750300
3,llama-3b,todo_decomposition,TODO-SAMPLE-004,0.600546
4,llama-3b,todo_decomposition,TODO-SAMPLE-005,0.854066


## 8. Pairwise Cosine Similarity

동일한 `task + sample_id`에 대해 모델별 생성 텍스트끼리 cosine similarity를 계산합니다. 이 값은 모델들이 서로 얼마나 비슷한 답을 냈는지 보는 지표입니다.

In [8]:
pair_source_df = text_df[text_df["has_candidate"]].copy().reset_index(drop=True)
pair_embeddings = embedding_model.encode(
    pair_source_df["candidate_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
pair_source_df["embedding_index"] = np.arange(len(pair_source_df))

pair_rows = []
for (task, sample_id), group in pair_source_df.groupby(["task", "sample_id"]):
    if len(group) < 2:
        continue
    group_rows = list(group.itertuples(index=False))
    for left, right in combinations(group_rows, 2):
        left_emb = pair_embeddings[left.embedding_index]
        right_emb = pair_embeddings[right.embedding_index]
        cosine = float(np.dot(left_emb, right_emb))
        pair_rows.append({
            "task": task,
            "sample_id": sample_id,
            "model_a": left.model_key,
            "model_b": right.model_key,
            "pairwise_cosine_similarity": cosine,
        })

pairwise_df = pd.DataFrame(pair_rows)
print(f"pairwise rows: {len(pairwise_df)}")
display(pairwise_df.head(10))

if not pairwise_df.empty:
    pairwise_summary_df = (
        pairwise_df
        .groupby(["task", "model_a", "model_b"], as_index=False)
        .agg(
            pairwise_cosine_mean=("pairwise_cosine_similarity", "mean"),
            pairwise_cosine_std=("pairwise_cosine_similarity", "std"),
            samples=("pairwise_cosine_similarity", "size"),
        )
        .sort_values(["task", "pairwise_cosine_mean"], ascending=[True, False])
    )
else:
    pairwise_summary_df = pd.DataFrame()

display(pairwise_summary_df.head(20))

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

pairwise rows: 1050


,task,sample_id,model_a,model_b,pairwise_cosine_similarity
0,character_message_generation,MSG-SAMPLE-001,llama-3b,gemma-3-4b-it,0.498119
1,character_message_generation,MSG-SAMPLE-001,llama-3b,midm-base,0.572816
2,character_message_generation,MSG-SAMPLE-001,llama-3b,midm-mini,0.457577
3,character_message_generation,MSG-SAMPLE-001,llama-3b,qwen-3b,0.528770
4,character_message_generation,MSG-SAMPLE-001,llama-3b,qwen-7b,0.562561
5,character_message_generation,MSG-SAMPLE-001,llama-3b,kanana-nano-2.1b-vllm,0.601441
6,character_message_generation,MSG-SAMPLE-001,gemma-3-4b-it,midm-base,0.482791
7,character_message_generation,MSG-SAMPLE-001,gemma-3-4b-it,midm-mini,0.459786
8,character_message_generation,MSG-SAMPLE-001,gemma-3-4b-it,qwen-3b,0.495781
9,character_message_generation,MSG-SAMPLE-001,gemma-3-4b-it,qwen-7b,0.498037


,task,model_a,model_b,pairwise_cosine_mean,pairwise_cosine_std,samples
19,character_message_generation,qwen-3b,qwen-7b,0.651813,0.146142,10
20,character_message_generation,qwen-7b,kanana-nano-2.1b-vllm,0.638090,0.152933,10
18,character_message_generation,qwen-3b,kanana-nano-2.1b-vllm,0.631463,0.050825,10
12,character_message_generation,midm-base,midm-mini,0.620659,0.100238,10
6,character_message_generation,llama-3b,kanana-nano-2.1b-vllm,0.614166,0.100497,10
4,character_message_generation,gemma-3-4b-it,qwen-7b,0.613721,0.130986,10
2,character_message_generation,gemma-3-4b-it,midm-mini,0.612320,0.127397,10
14,character_message_generation,midm-base,qwen-7b,0.607241,0.131358,10
17,character_message_generation,midm-mini,qwen-7b,0.602960,0.147375,10
0,character_message_generation,gemma-3-4b-it,kanana-nano-2.1b-vllm,0.598130,0.144570,10


## 9. Distinct-1 / Distinct-2

모델별/태스크별 생성 텍스트의 unigram, bigram 다양성을 계산합니다.

- `distinct_1 = unique unigrams / total unigrams`
- `distinct_2 = unique bigrams / total bigrams`

한국어 형태소 분석기는 추가하지 않고, 공백/기호 기준의 가벼운 토큰화를 사용합니다.

In [9]:
TOKEN_RE = re.compile(r"[가-힣A-Za-z0-9]+")


def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(normalize_text(text).lower())


def distinct_n(texts: Iterable[str], n: int) -> Tuple[float, int, int]:
    total = 0
    unique = set()
    for text in texts:
        tokens = tokenize(text)
        if len(tokens) < n:
            continue
        ngrams = list(zip(*(tokens[i:] for i in range(n))))
        total += len(ngrams)
        unique.update(ngrams)
    score = len(unique) / total if total else np.nan
    return score, len(unique), total

rows = []
for (model_key, task), group in text_df[text_df["has_candidate"]].groupby(["model_key", "task"]):
    candidate_texts = group["candidate_text"].tolist()
    d1, u1, t1 = distinct_n(candidate_texts, 1)
    d2, u2, t2 = distinct_n(candidate_texts, 2)
    rows.append({
        "model_key": model_key,
        "task": task,
        "samples": len(group),
        "distinct_1": d1,
        "distinct_1_unique": u1,
        "distinct_1_total": t1,
        "distinct_2": d2,
        "distinct_2_unique": u2,
        "distinct_2_total": t2,
    })

distinct_df = pd.DataFrame(rows).sort_values(["task", "model_key"]).reset_index(drop=True)
display(distinct_df)

,model_key,task,samples,distinct_1,distinct_1_unique,distinct_1_total,distinct_2,distinct_2_unique,distinct_2_total
0,gemma-3-4b-it,character_message_generation,10,0.951220,117,123,1.000000,113,113
1,kanana-nano-2.1b-vllm,character_message_generation,10,0.839378,162,193,0.989071,181,183
2,llama-3b,character_message_generation,10,0.809735,183,226,1.000000,216,216
3,midm-base,character_message_generation,10,0.925620,112,121,1.000000,111,111
4,midm-mini,character_message_generation,10,0.911290,113,124,1.000000,114,114
5,qwen-3b,character_message_generation,10,0.896774,139,155,0.993103,144,145
6,qwen-7b,character_message_generation,10,0.926230,113,122,1.000000,112,112
7,gemma-3-4b-it,feed_post_generation,10,0.754839,117,155,0.958621,139,145
8,kanana-nano-2.1b-vllm,feed_post_generation,10,0.681818,150,220,0.880952,185,210
9,llama-3b,feed_post_generation,10,0.824818,226,274,0.988636,261,264


## 10. 요약표 생성

In [10]:
semantic_summary_by_model = (
    valid_df
    .groupby("model_key", as_index=False)
    .agg(
        samples=("sample_id", "size"),
        embedding_cosine_mean=("embedding_cosine_similarity", "mean"),
        embedding_cosine_std=("embedding_cosine_similarity", "std"),
        bertscore_f1_mean=("bertscore_f1", "mean"),
        bertscore_f1_std=("bertscore_f1", "std"),
    )
)

distinct_summary_by_model = (
    distinct_df
    .groupby("model_key", as_index=False)
    .agg(
        distinct_1_mean=("distinct_1", "mean"),
        distinct_2_mean=("distinct_2", "mean"),
    )
)

model_summary_df = (
    semantic_summary_by_model
    .merge(distinct_summary_by_model, on="model_key", how="left")
    .sort_values(["embedding_cosine_mean", "bertscore_f1_mean"], ascending=False)
    .reset_index(drop=True)
)

task_summary_df = (
    valid_df
    .groupby(["model_key", "task"], as_index=False)
    .agg(
        samples=("sample_id", "size"),
        embedding_cosine_mean=("embedding_cosine_similarity", "mean"),
        bertscore_f1_mean=("bertscore_f1", "mean"),
    )
    .merge(distinct_df[["model_key", "task", "distinct_1", "distinct_2"]], on=["model_key", "task"], how="left")
    .sort_values(["task", "embedding_cosine_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

display(model_summary_df)
display(task_summary_df)

,model_key,samples,embedding_cosine_mean,embedding_cosine_std,bertscore_f1_mean,bertscore_f1_std,distinct_1_mean,distinct_2_mean
0,kanana-nano-2.1b-vllm,50,0.592240,0.221429,0.716254,0.100059,0.612140,0.807227
1,qwen-7b,50,0.586569,0.208662,0.717270,0.083338,0.717283,0.850601
2,midm-base,50,0.580641,0.240029,0.724790,0.093804,0.716844,0.897086
3,midm-mini,50,0.557607,0.193947,0.707688,0.077500,0.708480,0.858739
4,qwen-3b,50,0.556201,0.198657,0.716143,0.074635,0.672309,0.809076
5,gemma-3-4b-it,50,0.552196,0.220781,0.709906,0.091522,0.703023,0.866101
6,llama-3b,50,0.531102,0.172711,0.678923,0.058729,0.640752,0.782901


,model_key,task,samples,embedding_cosine_mean,bertscore_f1_mean,distinct_1,distinct_2
0,midm-base,character_message_generation,10,0.585691,0.721529,0.925620,1.000000
1,qwen-7b,character_message_generation,10,0.583279,0.716778,0.926230,1.000000
2,qwen-3b,character_message_generation,10,0.565333,0.711593,0.896774,0.993103
3,gemma-3-4b-it,character_message_generation,10,0.542318,0.707894,0.951220,1.000000
4,midm-mini,character_message_generation,10,0.534800,0.711285,0.911290,1.000000
5,kanana-nano-2.1b-vllm,character_message_generation,10,0.525358,0.705657,0.839378,0.989071
6,llama-3b,character_message_generation,10,0.458089,0.675302,0.809735,1.000000
7,qwen-7b,feed_post_generation,10,0.568199,0.698699,0.819549,0.934959
8,kanana-nano-2.1b-vllm,feed_post_generation,10,0.554630,0.705966,0.681818,0.880952
9,qwen-3b,feed_post_generation,10,0.550525,0.706387,0.756944,0.925373


## 11. 결과 저장

In [11]:
valid_df.to_csv(SEMANTIC_SCORE_PATH, index=False, encoding="utf-8-sig")
pairwise_df.to_csv(PAIRWISE_SCORE_PATH, index=False, encoding="utf-8-sig")
pairwise_summary_df.to_csv(PAIRWISE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
distinct_df.to_csv(DISTINCT_SCORE_PATH, index=False, encoding="utf-8-sig")
model_summary_df.to_csv(MODEL_SUMMARY_PATH, index=False, encoding="utf-8-sig")
task_summary_df.to_csv(TASK_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("저장 완료")
print("-", SEMANTIC_SCORE_PATH)
print("-", PAIRWISE_SCORE_PATH)
print("-", PAIRWISE_SUMMARY_PATH)
print("-", DISTINCT_SCORE_PATH)
print("-", MODEL_SUMMARY_PATH)
print("-", TASK_SUMMARY_PATH)

저장 완료
- /Users/areum/skn/final/data/evaluation/semantic_similarity_scores.csv
- /Users/areum/skn/final/data/evaluation/pairwise_cosine_similarity.csv
- /Users/areum/skn/final/data/evaluation/pairwise_cosine_summary.csv
- /Users/areum/skn/final/data/evaluation/distinct_scores.csv
- /Users/areum/skn/final/data/evaluation/semantic_distinct_model_summary.csv
- /Users/areum/skn/final/data/evaluation/semantic_distinct_task_summary.csv


## 12. 빠른 해석 가이드

- `embedding_cosine_similarity`: 기준 출력과 생성 출력의 의미 유사도입니다. 높을수록 기준과 의미가 가깝습니다.
- `bertscore_f1`: 기준 출력과 생성 출력의 contextual token similarity입니다. 높을수록 기준 문장과 표현/의미가 가깝습니다.
- `pairwise_cosine_similarity`: 같은 샘플에 대한 모델 간 답변 유사도입니다. 너무 높으면 모델들이 비슷한 답만 내고 있다는 뜻일 수 있고, 너무 낮으면 출력 일관성이 떨어질 수 있습니다.
- `distinct_1`, `distinct_2`: 생성 텍스트 다양성입니다. 높을수록 단어/구 조합이 다양합니다. 단, 정확도 지표가 아니므로 semantic score와 같이 봐야 합니다.